# 📦 Phase 1: PPTOnline Bulk Download (1.4M entries)

Downloads PPT files from the **pptonline** HuggingFace dataset (1,418,349 entries).

**Filters:** ❌ China/HK/Taiwan | ❌ Under 2MB | ❌ Duplicates | ✅ Magic bytes

In [ ]:
# Cell 1: Setup
!pip install -q datasets zstandard requests beautifulsoup4
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
warnings.filterwarnings('ignore')

DRIVE_DIR = '/content/drive/MyDrive/PPTX_Global_Collection'
SEEN_FILE = os.path.join(DRIVE_DIR, 'seen_tags.txt')
PROGRESS_FILE = os.path.join(DRIVE_DIR, 'hf_progress.json')
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Output: {DRIVE_DIR}')

In [ ]:
# Cell 2: Shared utilities
import hashlib, json, os, re, time, zipfile
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

MIN_SIZE = 2 * 1024 * 1024  # 2MB

CHINA_KW = ['chinese','china','hong kong','taiwan','beijing','shanghai',
            'mandarin','wuhan','guangzhou','shenzhen','nanjing','zhonghua']

def load_seen():
    tags = set()
    if os.path.exists(SEEN_FILE):
        with open(SEEN_FILE) as f:
            tags = {l.strip() for l in f if l.strip()}
    return tags

def save_tag(tag):
    with open(SEEN_FILE, 'a') as f: f.write(tag + '\n')

def is_china(text):
    if not text: return False
    t = text.lower()
    return any(kw in t for kw in CHINA_KW)

def has_chinese_chars(fp):
    try:
        with zipfile.ZipFile(fp, 'r') as z:
            for n in z.namelist():
                if 'slide' in n and n.endswith('.xml'):
                    d = z.read(n).decode('utf-8', errors='ignore')
                    if len(re.findall(r'[\u4e00-\u9fff]', d)) > 50: return True
    except: pass
    return False

def valid_ppt(fp):
    try:
        with open(fp, 'rb') as f: h = f.read(8)
        return h[:2] == b'PK' or h[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
    except: return False

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f: return json.load(f)
    return {'pptonline_idx': 0, 'downloaded': 0}

def save_progress(prog):
    with open(PROGRESS_FILE, 'w') as f: json.dump(prog, f)

session = requests.Session()
retry = Retry(total=3, backoff_factor=1, status_forcelist=[429,500,502,503,504])
session.mount('https://', HTTPAdapter(max_retries=retry))
session.mount('http://', HTTPAdapter(max_retries=retry))
session.headers.update({'User-Agent': 'Mozilla/5.0 Academic-Scraper/3.0'})

seen = load_seen()
progress = load_progress()
print(f'✅ Loaded {len(seen)} seen tags | Progress: {progress}')

In [ ]:
# Cell 3: Download engine
stats = {'checked': 0, 'downloaded': 0, 'small': 0, 'china': 0,
         'invalid': 0, 'dup': 0, 'errors': 0}

def download_pptx(url, title, source_prefix, source_id):
    global seen, stats
    tag = hashlib.sha1(url.encode()).hexdigest()[:10]
    if tag in seen: stats['dup'] += 1; return False

    if is_china(title): stats['china'] += 1; return False

    ext = '.ppt' if url.lower().endswith('.ppt') else '.pptx'
    safe = re.sub(r'[^\w\-]', '_', (title or 'untitled')[:40])
    fname = f'{tag}_{source_prefix}_{source_id}_{safe}{ext}'
    dest = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(dest): seen.add(tag); return False

    try:
        # HEAD check
        try:
            h = session.head(url, timeout=10, allow_redirects=True)
            cl = int(h.headers.get('Content-Length', 0))
            if 0 < cl < MIN_SIZE: stats['small'] += 1; return False
        except: pass

        r = session.get(url, timeout=90, stream=True)
        if not r.ok: stats['errors'] += 1; return False

        with open(dest, 'wb') as f:
            for chunk in r.iter_content(65536): f.write(chunk)

        sz = os.path.getsize(dest)
        if sz < MIN_SIZE: os.remove(dest); stats['small'] += 1; return False
        if not valid_ppt(dest): os.remove(dest); stats['invalid'] += 1; return False
        if has_chinese_chars(dest): os.remove(dest); stats['china'] += 1; return False

        seen.add(tag); save_tag(tag)
        stats['downloaded'] += 1
        return True
    except:
        if os.path.exists(dest): os.remove(dest)
        stats['errors'] += 1; return False

print('✅ Download engine ready')

In [ ]:
# Cell 4: pptonline (1.4M PPT metadata with constructed URLs)
from datasets import load_dataset
import traceback

# URL patterns to try for ppt-online.org downloads
URL_PATTERNS = [
    'https://ppt-online.org/download/{id}',
    'https://ppt-online.org/{id}/download',
    'https://en.ppt-online.org/download/{id}',
]

TARGET = 500000
print('🚀 pptonline dataset (1.4M entries)')
print('   Discovering download URL pattern first...')

# Step 1: Discover the correct URL pattern
try:
    ds_test = load_dataset('nyuuzyou/pptonline', split='train', streaming=True)
    test_ids = []
    for i, row in enumerate(ds_test):
        test_ids.append(row.get('id', i))
        if len(test_ids) >= 3: break

    working_pattern = None
    for pattern in URL_PATTERNS:
        for tid in test_ids:
            url = pattern.format(id=tid)
            try:
                r = session.head(url, timeout=10, allow_redirects=True)
                if r.status_code == 200:
                    ct = r.headers.get('Content-Type', '')
                    if 'presentation' in ct or 'octet' in ct or 'powerpoint' in ct:
                        working_pattern = pattern
                        print(f'  ✅ Found working URL pattern: {pattern}')
                        break
            except: continue
        if working_pattern: break

    if not working_pattern:
        # Try downloading the page to find download link
        print('  ⚠️  Direct download patterns failed. Trying page scraping...')
        from bs4 import BeautifulSoup
        for tid in test_ids:
            for base in ['https://ppt-online.org/', 'https://en.ppt-online.org/']:
                try:
                    r = session.get(f'{base}{tid}', timeout=15)
                    if r.ok:
                        soup = BeautifulSoup(r.text, 'html.parser')
                        dl_link = soup.find('a', href=True, string=lambda t: t and 'download' in t.lower() if t else False)
                        if not dl_link:
                            dl_link = soup.find('a', class_=lambda c: c and 'download' in str(c).lower() if c else False)
                        if dl_link:
                            href = dl_link['href']
                            if not href.startswith('http'): href = base.rstrip('/') + href
                            working_pattern = href.replace(str(tid), '{id}')
                            print(f'  ✅ Discovered download URL: {working_pattern}')
                            break
                except: continue
            if working_pattern: break

    if not working_pattern:
        print('  ❌ Could not find working download URL pattern for ppt-online.org')
        print('     The site may be down or requires different access.')
    else:
        # Step 2: Bulk download
        print(f'\n  Starting bulk download with pattern: {working_pattern}')
        ds = load_dataset('nyuuzyou/pptonline', split='train', streaming=True)
        start_idx = progress.get('pptonline_idx', 0)
        count = 0
        batch_count = 0

        for i, row in enumerate(ds):
            if i < start_idx: continue
            if stats['downloaded'] >= TARGET: break
            stats['checked'] += 1

            rid = row.get('id', i)
            title = row.get('title', 'untitled')
            category = row.get('category', '')

            if is_china(title) or is_china(category):
                stats['china'] += 1; continue

            url = working_pattern.format(id=rid)
            if download_pptx(url, title, 'pon', str(rid)):
                count += 1
                if count % 25 == 0:
                    print(f'  ✅ [{count}] Downloaded | Checked: {stats["checked"]} | '
                          f'Small: {stats["small"]} | Dup: {stats["dup"]}')

            batch_count += 1
            if batch_count % 1000 == 0:
                progress['pptonline_idx'] = i
                progress['downloaded'] = stats['downloaded']
                save_progress(progress)
                print(f'  💾 Progress saved at index {i}')

            if batch_count % 100 == 0: time.sleep(1)  # Rate limit

        progress['pptonline_idx'] = i if 'i' in dir() else start_idx
        save_progress(progress)
        print(f'\n✅ Complete: {count} files from pptonline')

except Exception as e:
    print(f'❌ Error: {e}')
    traceback.print_exc()
    save_progress(progress)

print(f'\n📊 Final stats: {stats}')

In [ ]:
# Cell 5: Final Summary
import glob
files = glob.glob(os.path.join(DRIVE_DIR,'*.pptx')) + glob.glob(os.path.join(DRIVE_DIR,'*.ppt'))
total_gb = sum(os.path.getsize(f) for f in files) / (1024**3)
pon = len([f for f in files if '_pon_' in f])
print(f'📊 FINAL SUMMARY')
print(f'   pptonline:  {pon} files')
print(f'   Other:      {len(files) - pon} files')
print(f'   TOTAL:      {len(files)} files | {total_gb:.2f} GB')
print(f'   Location:   Google Drive > PPTX_Global_Collection')
print(f'\n📊 Filter stats: {stats}')